# Session 11: Exception Handling and Reliable Python

**Python Fundamentals | Session 11 of 14 | Guided learning: 30 minutes**

The Learning Assistant must handle invalid study data without losing valid information. Follow **Predict → Run → Modify → Verify → Reflect**. This notebook uses only the standard library and every executable cell is safe to run from top to bottom.

## 1. When Python Encounters a Problem (3 minutes)

**CONCEPT:** A syntax error means Python cannot parse the code at all. A runtime exception happens while valid code is running. A logical error runs successfully but produces the wrong result.

Examples include `int("forty")` → `ValueError`, `60 / 0` → `ZeroDivisionError`, `durations[10]` → `IndexError`, `"45" + 5` → `TypeError`, and `{"topic": "Python"}["duration"]` → `KeyError`. An invalid indentation example is best kept in Markdown because it cannot run.

**PREDICT:** Which exception type belongs to each operation above? A quick classification is: invalid value, impossible arithmetic, missing position, incompatible types and missing dictionary key.

In [ ]:
def exception_name(operation):
    try:
        operation()
    except Exception as error:
        return type(error).__name__
    return "no exception"

durations = [30, 45]
classification = {
    "convert text": exception_name(lambda: int("forty")),
    "divide by zero": exception_name(lambda: 60 / 0),
    "missing position": exception_name(lambda: durations[10]),
    "incompatible types": exception_name(lambda: "45" + 5),
    "missing key": exception_name(lambda: {"topic": "Python"}["duration"]),
}
print(classification)
assert classification["convert text"] == "ValueError"
assert classification["divide by zero"] == "ZeroDivisionError"
assert classification["missing position"] == "IndexError"
assert classification["incompatible types"] == "TypeError"
assert classification["missing key"] == "KeyError"

**MODIFY 1:** Add another safe classification, such as `int("30.5")`, and predict its exception before running it.

**VERIFY:** The operation may be valid Python syntax even when its runtime value is unacceptable. A logical mistake, such as calculating `duration / 100` instead of `/ 60`, would not raise an exception but would still be wrong.

**REFLECT:** Why should syntax errors and runtime exceptions be investigated differently?

## 2. Catching Expected Errors with `try`/`except` (4 minutes)

**CONCEPT:** The `try` block contains an operation that may fail. `except ValueError` handles one expected kind of failure. This demonstration returns `None` for invalid text; a full application would usually retry or report the problem to its validation layer.

**PREDICT:** What should `safe_minutes()` return for `"45"`, `"abc"`, `"0"`, `"-10"` and `"30.5"`?

In [ ]:
def safe_minutes(value):
    try:
        minutes = int(value)
    except ValueError:
        return None
    if minutes <= 0:
        return None
    return minutes

safe_results = [safe_minutes(value) for value in ["45", "abc", "0", "-10", "30.5"]]
print(safe_results)
assert safe_minutes("45") == 45
assert safe_minutes("abc") is None
assert safe_minutes("0") is None
assert safe_minutes("-10") is None
assert safe_minutes("30.5") is None

**MODIFY 2:** Add one more invalid input, such as `"many"`, to the test list. The expected result is `None`.

**VERIFY:** Conversion and positive-value validation are separate decisions. `int("30.5")` raises `ValueError`, while `int("-10")` succeeds but the positive-value rule rejects it.

**REFLECT:** Why is catching only `ValueError` clearer here than catching every possible exception?

## 3. Common Exceptions and Precise Handling (3 minutes)

**CONCEPT:** Different errors may need different responses. `ValueError` means the value has an unsuitable form, `ZeroDivisionError` means a calculation is impossible, `TypeError` means incompatible types were used, `KeyError` means a dictionary key is absent, and `IndexError` means a sequence position is outside its range.

**PREDICT:** Which handler will run for invalid text, and which will run when the total is divided by zero?

In [ ]:
def average_duration(total, count):
    try:
        return int(total) / int(count)
    except ValueError:
        return "Total and count must be numbers"
    except ZeroDivisionError:
        return "At least one activity is required"

print(average_duration("90", "2"))
print(average_duration("many", "2"))
print(average_duration("90", "0"))
assert average_duration("90", "2") == 45.0
assert average_duration("many", "2") == "Total and count must be numbers"
assert average_duration("90", "0") == "At least one activity is required"

**MODIFY 3:** Change the count to `"three"` and predict which message appears. Match these situations: missing dictionary key → `KeyError`; invalid list position → `IndexError`; invalid numeric text → `ValueError`.

**VERIFY:** Precise handlers preserve useful meaning. A bare `except:` or broad `except Exception:` can hide programming defects that should be fixed rather than treated as input.

## 4. Raising Exceptions to Protect Data (4 minutes)

**CONCEPT:** The Session 10 property protected a `StudySession` duration. Raising `ValueError` stops invalid state from entering the object. The constructor also uses the property, so initial values are checked.

**PREDICT:** Why should `True` be rejected even though Python treats `bool` as a subclass of `int`? A boolean describes yes/no, not a number of minutes.

In [ ]:
class StudySession:
    def __init__(self, topic, duration):
        self.topic = topic
        self.duration = duration

    @property
    def duration(self):
        return self._duration

    @duration.setter
    def duration(self, value):
        if type(value) is not int or value <= 0:
            raise ValueError("Duration must be a positive integer")
        self._duration = value

session = StudySession("Python", 45)
try:
    session.duration = -10
except ValueError as error:
    print(error)

print(session.duration)
assert session.duration == 45
assert session.topic == "Python"

In [ ]:
def update_is_rejected(session, value):
    previous = session.duration
    try:
        session.duration = value
    except ValueError:
        return session.duration == previous
    return False

for invalid in (0, -5, "45", 30.5, True):
    assert update_is_rejected(session, invalid)
assert session.duration == 45
print("All invalid updates were rejected without corrupting state.")

**MODIFY 4:** Test two more invalid values, such as `None` and `"sixty"`. Confirm the original duration is still 45.

**REFLECT:** What is the difference between raising an exception and catching it? Raising reports that the current operation cannot continue; catching decides where and how to handle that signal.

## 5. Understanding `else` and `finally` (4 minutes)

**CONCEPT:** `try` attempts the operation, `except` handles a matching exception, `else` runs only when `try` succeeds, and `finally` runs whether an exception occurs or not, including while an exception is propagating. `finally` reports cleanup or completion of the attempt; it does not guarantee success. Do not return from `finally`, because that can suppress an exception or override another return value.

**PREDICT:** What order of messages will `trace_conversion("45")`, `trace_conversion("abc")` and `trace_conversion("0")` return?

In [ ]:
def trace_conversion(value):
    messages = ["try started"]
    try:
        minutes = int(value)
    except ValueError:
        messages.append("except handled invalid text")
    else:
        messages.append("else conversion succeeded")
        if minutes <= 0:
            messages.append("else found a non-positive value")
        else:
            messages.append("else found a positive value")
    finally:
        messages.append("finally always ran")
    return messages

valid_trace = trace_conversion("45")
text_trace = trace_conversion("abc")
zero_trace = trace_conversion("0")
print(valid_trace)
print(text_trace)
print(zero_trace)
assert valid_trace == ["try started", "else conversion succeeded", "else found a positive value", "finally always ran"]
assert text_trace == ["try started", "except handled invalid text", "finally always ran"]
assert zero_trace == ["try started", "else conversion succeeded", "else found a non-positive value", "finally always ran"]

**MODIFY 5:** Add another test case, such as `"-3"`, and predict its message order. It converts successfully, so `else` runs, but it is non-positive.

**VERIFY:** `finally` appears in every returned trace. It means the finalisation step ran, not that conversion succeeded.

## 6. Exception Propagation Between Functions (3 minutes)

**CONCEPT:** An exception can be detected in one function and handled by its caller. `create_study_session()` deliberately does not catch `ValueError`; the validation signal travels to the calling function.

```text
StudySession validation → create_study_session() → caller handles ValueError
```

**PREDICT:** Will the wrapper return an object for duration `"45"`? What will happen for `"bad"`?

In [ ]:
def create_study_session(topic, duration):
    return StudySession(topic, duration)

def caller_builds_session(topic, duration):
    try:
        return create_study_session(topic, duration)
    except ValueError:
        return None

valid_created = caller_builds_session("Python", 45)
invalid_created = caller_builds_session("Python", "bad")
print(valid_created.topic)
print(invalid_created)
assert valid_created.duration == 45
assert invalid_created is None

**MODIFY 6:** Change the invalid duration to `0` or `-20` and repeat the experiment. The error is detected by the model but handled by the caller.

**REFLECT:** When is it better to let an exception propagate? Let it travel to a layer that has enough context to respond appropriately, rather than hiding it at the point of detection.

## 7. Reliable Input Validation Without `input()` (3 minutes)

**CONCEPT:** We can simulate repeated user attempts with a predefined list. The same loop could later read values from `input()`, but this notebook remains deterministic and safe for Run all.

**PREDICT:** Which value will `first_valid_duration(["abc", "-5", "45"])` accept? It should skip the first two and return 45.

In [ ]:
def first_valid_duration(attempts):
    for value in attempts:
        try:
            minutes = int(value)
        except ValueError:
            continue
        if minutes > 0:
            return minutes
    return None

attempts = ["abc", "-5", "45"]
print(first_valid_duration(attempts))
assert first_valid_duration(attempts) == 45
assert first_valid_duration(["x", "0", "-3"]) is None


**MODIFY 7:** Change the attempts list so every value is invalid, for example `["zero", "0", "-1"]`. The function should return `None` after checking all attempts.

**VERIFY:** The loop preserves the first valid value and does not create a `StudySession` until validation succeeds.

## 8. Debug AI-Generated Exception Handling (4 minutes)

**DEBUG 1 — Bare `except`:**
```python
try:
    minutes = int(value)
except:
    minutes = 0
```
This hides every error, including programming defects. Catch the expected `ValueError` specifically.

**DEBUG 2 — Silently replacing invalid input:** Returning or storing zero changes the user's data and makes invalid input look valid. Return `None` or retry instead.

**DEBUG 3 — Returning from `finally`:**
```python
def unsafe(value):
    try:
        return int(value)
    finally:
        return 0
```
The `finally` return overrides the successful result and suppresses conversion errors.

**DEBUG 4 — Constructor hides validation:** Catching a model's own `ValueError` and continuing can create an invalid object or a misleading success. Let validation propagate to the caller.

**PREDICT:** Which repair preserves the original error meaning while handling expected input problems?

In [ ]:
def precise_conversion(value):
    try:
        return int(value)
    except ValueError:
        return None

def safe_finalisation(value):
    result = None
    try:
        result = int(value)
    except ValueError:
        result = None
    finally:
        print("Conversion attempt finished")
    return result

assert precise_conversion("45") == 45
assert precise_conversion("bad") is None
assert safe_finalisation("30") == 30
assert safe_finalisation("bad") is None
assert caller_builds_session("Python", -1) is None
print("Focused repairs preserve expected error behaviour.")

**VERIFY:** Specific handling makes expected input failures understandable without masking unrelated defects. Not every exception should be caught; unexpected programming errors should remain visible during development.

**REFLECT:** Why is `except Exception:` usually inappropriate for simple input validation? It catches far more than the expected conversion error and can hide defects.

## 9. Mini Integration Challenge and Reflection (2 minutes)

**CONCEPT:** A reliable factory converts text, validates a positive duration through `StudySession`, catches only expected `ValueError` failures and preserves valid objects. The starter below is complete so Run all succeeds; learners can replace the TODO body in their own copy.

**PREDICT:** What should `build_session("Python", "45")` and `build_session("Python", "bad")` return?

In [ ]:
def build_session(topic, duration_text):
    # TODO: convert text, construct StudySession, and handle expected ValueError.
    try:
        duration = int(duration_text)
        return StudySession(topic, duration)
    except ValueError:
        return None

built_session = build_session("Python", "45")
rejected_session = build_session("Python", "bad")
zero_session = build_session("Python", "0")
assert built_session.topic == "Python"
assert built_session.duration == 45
assert rejected_session is None
assert zero_session is None
print(built_session.topic, built_session.duration)
print(rejected_session, zero_session)

**MODIFY:** Test two additional text values and decide whether they should create an object. Do not catch unrelated programming defects in the factory.

**REFLECT:**

1. What is the difference between raising and catching an exception? Raising signals a failed operation; catching handles that signal at a suitable layer.
2. Why should we catch specific exceptions? Specific handlers preserve the meaning of different failures.
3. When does the `else` block execute? Only when the `try` block completes without an exception.
4. Why does `finally` not guarantee success? It runs after both successful and failed attempts.
5. When is it better to let an exception propagate? When a caller has the context needed to handle it or when the defect is unexpected.

**VERIFY:** The notebook has no unhandled executable exceptions, preserves valid object state and uses assertions for the integration behaviour.

**Handle expected errors gracefully. Never hide unexpected failures.**